# Model 1: HAM10000 Training

This notebook trains a ResNet152V2 model on the HAM10000 dataset.
**Optimized for Google Colab T4 GPU**

In [ ]:
# Mount Google Drive to save model
from google.colab import drive
drive.mount('/content/drive')

# Create model directory in Drive
import os
MODEL_SAVE_DIR = '/content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {MODEL_SAVE_DIR}")

Mounted at /content/drive
Models will be saved to: /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models


In [ ]:
# Install dependencies
!pip install kagglehub -q

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.applications import ResNet152V2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Set seeds
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

TensorFlow: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Download HAM10000
print("Downloading HAM10000...")
HAM_PATH = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print(f"HAM10000 Path: {HAM_PATH}")

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
HAM10000 Path: /kaggle/input/skin-cancer-mnist-ham10000


In [ ]:
# Explore HAM10000 structure
import glob
import shutil

# Find metadata and images
metadata_files = glob.glob(os.path.join(HAM_PATH, '**/*.csv'), recursive=True)
image_dirs = glob.glob(os.path.join(HAM_PATH, '**/*images*'), recursive=True)

print(f"Found {len(metadata_files)} metadata files")
print(f"Found {len(image_dirs)} image directories")

# Load metadata
metadata_file = [f for f in metadata_files if 'metadata' in f.lower()][0]
df = pd.read_csv(metadata_file)
print(f"\nLoaded metadata: {len(df)} images")
print(f"\nClasses:\n{df['dx'].value_counts()}")

Found 5 metadata files
Found 4 image directories

Loaded metadata: 10015 images

Classes:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [ ]:
# Create organized directory structure
WORK_DIR = '/content/ham10000_organized'
TRAIN_DIR = os.path.join(WORK_DIR, 'train')
TEST_DIR = os.path.join(WORK_DIR, 'test')

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# Create class folders
classes = df['dx'].unique()
for cls in classes:
    os.makedirs(os.path.join(TRAIN_DIR, cls), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, cls), exist_ok=True)

print(f"Created folders for {len(classes)} classes")

Created folders for 7 classes


In [ ]:
# Organize images into train/test
from sklearn.model_selection import train_test_split

# Find all image files
all_images = []
for img_dir in image_dirs:
    if os.path.isdir(img_dir):
        all_images.extend(glob.glob(os.path.join(img_dir, '*.jpg')))

print(f"Found {len(all_images)} total images")

# Create image_id to path mapping
image_map = {}
for img_path in all_images:
    img_id = os.path.basename(img_path).replace('.jpg', '')
    image_map[img_id] = img_path

# Split by class
train_count = 0
test_count = 0

for cls in tqdm(classes, desc="Organizing images"):
    class_df = df[df['dx'] == cls]

    # Split 80/20
    train_ids, test_ids = train_test_split(
        class_df['image_id'].values,
        test_size=0.2,
        random_state=42
    )

    # Copy train images
    for img_id in train_ids:
        if img_id in image_map:
            src = image_map[img_id]
            dst = os.path.join(TRAIN_DIR, cls, f"{img_id}.jpg")
            shutil.copy2(src, dst)
            train_count += 1

    # Copy test images
    for img_id in test_ids:
        if img_id in image_map:
            src = image_map[img_id]
            dst = os.path.join(TEST_DIR, cls, f"{img_id}.jpg")
            shutil.copy2(src, dst)
            test_count += 1

print(f"\nOrganized {train_count} training images")
print(f"Organized {test_count} test images")

Found 20030 total images


Organizing images: 100%|██████████| 7/7 [01:30<00:00, 12.92s/it]


Organized 8010 training images
Organized 2005 test images


In [ ]:
# Setup data generators
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print(f"Classes: {class_names}")
print(f"Number of classes: {num_classes}")

Found 8010 images belonging to 7 classes.
Found 2005 images belonging to 7 classes.
Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Number of classes: 7


In [ ]:
# Build model
base_model = ResNet152V2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model built successfully!")

234545216/234545216 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Model built successfully!


In [ ]:
# Setup callbacks
callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(MODEL_SAVE_DIR, 'ham10000_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# Phase 1: Train frozen base
print("="*50)
print("PHASE 1: Training with frozen base")
print("="*50)

history1 = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=callbacks,
    verbose=1
)

PHASE 1: Training with frozen base


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 681ms/step - accuracy: 0.5581 - loss: 1.4156
Epoch 1: val_accuracy improved from -inf to 0.71771, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/ham10000_best.keras
251/251 ━━━━━━━━━━━━━━━━━━━━ 241s 832ms/step - accuracy: 0.5583 - loss: 1.4146 - val_accuracy: 0.7177 - val_loss: 0.7897 - learning_rate: 0.0010
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 625ms/step - accuracy: 0.6684 - loss: 0.9568
Epoch 2: val_accuracy improved from 0.71771 to 0.73367, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/ham10000_best.keras
251/251 ━━━━━━━━━━━━━━━━━━━━ 178s 710ms/step - accuracy: 0.6685 - loss: 0.9567 - val_accuracy: 0.7337 - val_loss: 0.7709 - learning_rate: 0.0010
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 603ms/step - accuracy: 0.6956 - loss: 0.8496
Epoch 3: val_accuracy improved from 0.73367 to 0.74214, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/ham10000_best.keras

In [ ]:
# Phase 2: Fine-tune
print("="*50)
print("PHASE 2: Fine-tuning")
print("="*50)

base_model.trainable = True

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=15,
    validation_data=test_generator,
    callbacks=callbacks,
    verbose=1
)

PHASE 2: Fine-tuning
Epoch 1/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7198 - loss: 0.7865
Epoch 1: val_accuracy improved from 0.75511 to 0.76608, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/ham10000_best.keras
251/251 ━━━━━━━━━━━━━━━━━━━━ 442s 1s/step - accuracy: 0.7199 - loss: 0.7863 - val_accuracy: 0.7661 - val_loss: 0.8310 - learning_rate: 1.0000e-04
Epoch 2/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 872ms/step - accuracy: 0.7762 - loss: 0.6294
Epoch 2: val_accuracy did not improve from 0.76608
251/251 ━━━━━━━━━━━━━━━━━━━━ 235s 937ms/step - accuracy: 0.7762 - loss: 0.6293 - val_accuracy: 0.7646 - val_loss: 0.6613 - learning_rate: 1.0000e-04
Epoch 3/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 821ms/step - accuracy: 0.7939 - loss: 0.5451
Epoch 3: val_accuracy improved from 0.76608 to 0.78953, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/ham10000_best.keras
251/251 ━━━━━━━━━━━━━━━━━━━━ 230s 915ms/step - accuracy: 0.7939 - loss: 

In [ ]:
# Evaluate
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")

63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 245ms/step - accuracy: 0.7477 - loss: 0.6991

Test Accuracy: 84.24%


In [ ]:
# Save final model and metadata
model.save(os.path.join(MODEL_SAVE_DIR, 'ham10000_final.keras'))

# Save class names
with open(os.path.join(MODEL_SAVE_DIR, 'ham10000_classes.json'), 'w') as f:
    json.dump(class_names, f)

# Save model info
model_info = {
    'dataset': 'HAM10000',
    'num_classes': num_classes,
    'classes': class_names,
    'test_accuracy': float(test_accuracy),
    'image_type': 'dermatoscopic',
    'input_size': IMG_SIZE
}

with open(os.path.join(MODEL_SAVE_DIR, 'ham10000_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print("\n✅ Model saved to Google Drive!")
print(f"Location: {MODEL_SAVE_DIR}")


✅ Model saved to Google Drive!
Location: /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models
